# Make Demo Samples

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

from nd_aligner.config.ndaligner.training_module_config import NDAlignerTrainingModuleConfigs
from nd_aligner.config.utils.io import load_config
from nd_aligner.models.ndaligner import init_nd_aligner_training_module

.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
.venv/lib/python3.14/site-packages/webrtcvad.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## INIT Models

In [ ]:
# VCTK + LibriSpeech

aligner_training_module_cfg_path = "../checkpoints/ndaligner/v2.3/VCTK+LibriSpeech/model_config.json"
aligner_training_module_ckpt_path = "../checkpoints/ndaligner/v2.3/VCTK+LibriSpeech/best_step_timit_bae_0.017166_step_267000_epoch_9.pth"

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(
    config=model_config,
    device=device,
)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

Loading nested state_dict from key 'model' in checkpoints/ndaligner/v2.3/VCTK+LibriSpeech/best_step_timit_bae_0.017166_step_267000_epoch_9.pth
✅ All weights matched perfectly.
Checkpoint loading process finished.


In [4]:
import re
from pathlib import Path

import soundfile as sf

RAW_DIR = Path("raw_samples")
OUT_DIR = Path("processed_grids")
CORPORA = ("TIMIT", "Buckeye", "LibriSpeech")

INCLUDE_SPACE_TOKEN = False

# Both corpora store transcripts as `<start_sample> <end_sample> <text>`.
_LEADING_INDICES = re.compile(r"^\s*\d+\s+\d+\s+")


def read_transcript(path: Path) -> str:
    """Read a transcript file, dropping the leading sample indices."""
    line = path.read_text(encoding="utf-8", errors="replace").strip()
    return _LEADING_INDICES.sub("", line).strip()


def stage_audio(source: Path, target: Path) -> None:
    """
    Copy audio as plain PCM wav.

    TIMIT ships NIST SPHERE files under a `.WAV` name, which ffmpeg cannot read
    when the demo page is built, so everything is normalized here.
    """
    audio, sample_rate = sf.read(source)
    sf.write(target, audio, sample_rate, subtype="PCM_16")


def collect(corpus_dir: Path) -> list[tuple[Path, Path]]:
    """Pair every audio file in a directory with its transcript."""
    pairs: list[tuple[Path, Path]] = []

    for audio_path in sorted(corpus_dir.iterdir()):
        if audio_path.suffix.lower() != ".wav":
            continue

        transcript_path = audio_path.with_suffix(".TXT")
        if not transcript_path.exists():
            transcript_path = audio_path.with_suffix(".txt")

        if transcript_path.exists():
            pairs.append((audio_path, transcript_path))
        else:
            print(f"  no transcript for {audio_path.name}, skipping")

    return pairs


for corpus in CORPORA:
    corpus_dir = RAW_DIR / corpus
    out_dir = OUT_DIR / corpus
    out_dir.mkdir(parents=True, exist_ok=True)

    pairs = collect(corpus_dir)
    print(f"\n{corpus}: {len(pairs)} utterances")

    if not pairs:
        continue

    staged: list[Path] = []
    texts: list[str] = []

    for audio_path, transcript_path in pairs:
        target = out_dir / f"{audio_path.stem}.wav"
        stage_audio(audio_path, target)

        text = read_transcript(transcript_path)
        staged.append(target)
        texts.append(text)

        print(f"  {audio_path.stem}: {text}")

    written = aligner.align_to_textgrid(
        wav_paths=staged,
        texts=texts,
        output_dir=out_dir,
        include_space_token=INCLUDE_SPACE_TOKEN,
    )

    for path in written:
        print(f"  wrote {path.name}")


TIMIT: 3 utterances
  SI1103: Their work mirrors the mentality of the psychopath, rootless and irresponsible.
  SI1610: For an instant the old aunt felt something indefinable flash through her smile.
  SI749: Fold into whipped cream and add a dash of salt and sprinkling of paprika.
  wrote SI1103.TextGrid
  wrote SI1610.TextGrid
  wrote SI749.TextGrid

Buckeye: 3 utterances
  s0201a_chunk_0013: a number of years ago when we were traveling a lot and everything else i uh we did absentee voting
  s1401a_chunk_0004: i think it's a great city although the last three months i've threatened to move on more than one occasion because of all the traffic congestion and all of the road construction you just can't get anywhere
  s3001a_chunk_0014: no no um there's a few townie bars but college students aren't real welcome there so
  wrote s0201a_chunk_0013.TextGrid
  wrote s1401a_chunk_0004.TextGrid
  wrote s3001a_chunk_0014.TextGrid

LibriSpeech: 3 utterances
  4852-28311-0024: he had never seen 

In [5]:
for corpus in CORPORA:
    out_dir = OUT_DIR / corpus
    grids = sorted(out_dir.glob("*.TextGrid"))
    print(f"{corpus}: {len(grids)} TextGrids in {out_dir}")

TIMIT: 3 TextGrids in processed_grids/TIMIT
Buckeye: 3 TextGrids in processed_grids/Buckeye
LibriSpeech: 3 TextGrids in processed_grids/LibriSpeech
